# Density Extraction Notebook

This notebook is responsible for extracting density information for some ingredients. The data is scraped from King Arthur Baking's ingredient weight chart, which can be found at the following URL:

> "https://www.kingarthurbaking.com/learn/ingredient-weight-chart"

Once the data is scraped, the data is then saved to a CSV file.

In [1]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
from ingredient_parser import parse_ingredient
from ingredient_parser.dataclasses import IngredientAmount
from pint import UnitRegistry

In [2]:
url = "https://www.kingarthurbaking.com/learn/ingredient-weight-chart"

In [3]:
# defining default headers to avoid being blocked by the website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

In [4]:
response = requests.get(url, headers = headers)

The `BeautifulSoup` library is used for web scraping, and the `pandas` library is used for easy data manipulation.

In [5]:
soup = BeautifulSoup(response.content, "html.parser")

In [6]:
table = soup.find("table")

In [7]:
assert table is not None, "Could not find the table on the webpage"

data = []

headers = [th.text.strip() for th in table.find_all("th")]

assert len(headers) == 4

for row in table.find_all("tr") [1:]: # skipping header row, only want data here
    cells = row.find_all("td")

    if len(cells) > 0:
        data.append([re.sub(r"[^\x20-\x7E]", "", cell.text.strip()) for cell in cells]) # removing any weird, non-ASCII characters from the cell text

In [8]:
data_df = pd.DataFrame(data, columns = headers)

In [9]:
data_df

,Ingredient,Volume,Ounces,Grams
0,'00' Pizza Flour,1 cup,4,116
1,Agave syrup,1/4 cup,3,84
2,All-Purpose Baking Mix,1 cup,4 1/4,120
3,All-Purpose Flour,1 cup,4 1/4,120
4,Almond butter,1/4 cup,2 1/3,68
...,...,...,...,...
320,Yeast (instant),2 1/4 teaspoons,1/4,7
321,Yeast (instant),1 tablespoon,1/3,9
322,Yogurt,1 cup,8,227
323,Yuletide Cheer Fruit Blend,1 cup,4 1/2,130


There is no need to keep the "Ounces" column, as the data should only deal with metric units.

In [10]:
data_df = data_df.drop(columns = ["Ounces"])

The `pint` library provides an easy way to convert between different units of measurement, which is then useful for calculating densities.

In [11]:
ureg = UnitRegistry()

In [12]:
def calculate_volume(amount: IngredientAmount, ingredient_name: str | None = None) -> float:
    """
    Calculates the volume of an ingredient given its amount and name. The volume is returned in cubic centimeters (cm^3)

    If the unit of the amount is not a volume unit, then the function will attempt to handle some common cases manually based on the ingredient name. If the ingredient name is not recognized, then the function will return 0

    :param amount: the amount of the ingredient, which includes a quantity and a unit
    :type amount: IngredientAmount
    :param ingredient_name: the name of the ingredient, which is used for handling some common cases manually when the unit of the amount is not a volume unit (default = None)
    :type ingredient_name: str | None

    :return: the volume of the ingredient in cubic centimeters (cm^3)
    :rtype: float
    """

    if amount.unit not in ["cup", "tablespoon", "teaspoon"]:
        # handling the uncaught units manually
        if ingredient_name == "Egg (fresh)":
            return 50.0
        elif ingredient_name == "Egg white (fresh)":
            return 30.0
        elif ingredient_name == "Egg yolk (fresh)":
            return 20.0
        elif ingredient_name == "Garlic (cloves, in skin for roasting)":
            return 90.0

        return 0

    volume = ureg.Quantity(amount.quantity, amount.unit).to("cm^3")

    return float(volume.magnitude)

The `ingredient-parser` library is used to parse the volume strings into `pint` quantities, which can then be easily converted to a common unit for density calculation. It handles cases like "1 cup", "2 1/4 tablespoons", etc., and provides structured access to both the quantity and the unit.

In [ ]:
data_df ["parsed amounts"] = data_df ["Volume"].apply(lambda volume: parse_ingredient(volume).amount [0]) # type: ignore

In [ ]:
data_df ["volume (cm^3)"] = data_df.apply(lambda row: calculate_volume(row ["parsed amounts"], row ["Ingredient"]), axis = 1)

In [15]:
data_df

,Ingredient,Volume,Grams,parsed amounts,volume (cm^3)
0,'00' Pizza Flour,1 cup,116,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236
1,Agave syrup,1/4 cup,84,"IngredientAmount(quantity=Fraction(1, 4), quan...",59.147059
2,All-Purpose Baking Mix,1 cup,120,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236
3,All-Purpose Flour,1 cup,120,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236
4,Almond butter,1/4 cup,68,"IngredientAmount(quantity=Fraction(1, 4), quan...",59.147059
...,...,...,...,...,...
320,Yeast (instant),2 1/4 teaspoons,7,"IngredientAmount(quantity=Fraction(9, 4), quan...",11.090074
321,Yeast (instant),1 tablespoon,9,"IngredientAmount(quantity=Fraction(1, 1), quan...",14.786765
322,Yogurt,1 cup,227,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236
323,Yuletide Cheer Fruit Blend,1 cup,130,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236


Some rows have their "Grams" attribute in the form of a range ("x to y" grams). In these cases, the average of the range is taken as the value for the "Grams" column.

In [16]:
parsed_grams = []

for grams in data_df ["Grams"]:
    if "to" in grams:
        x, y = grams.split("to")
        parsed_grams.append((float(x) + float(y)) / 2)
    else:
        parsed_grams.append(float(grams))

data_df ["Grams"] = parsed_grams

Finally, the density is calculated in g/cm<sup>3</sup> and saved to a CSV file for later use.

In [17]:
data_df ["density (g/cm^3)"] = data_df ["Grams"].astype(float) / data_df ["volume (cm^3)"].astype(float)

In [18]:
data_df

,Ingredient,Volume,Grams,parsed amounts,volume (cm^3),density (g/cm^3)
0,'00' Pizza Flour,1 cup,116.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236,0.490303
1,Agave syrup,1/4 cup,84.0,"IngredientAmount(quantity=Fraction(1, 4), quan...",59.147059,1.420189
2,All-Purpose Baking Mix,1 cup,120.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236,0.507210
3,All-Purpose Flour,1 cup,120.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236,0.507210
4,Almond butter,1/4 cup,68.0,"IngredientAmount(quantity=Fraction(1, 4), quan...",59.147059,1.149677
...,...,...,...,...,...,...
320,Yeast (instant),2 1/4 teaspoons,7.0,"IngredientAmount(quantity=Fraction(9, 4), quan...",11.090074,0.631195
321,Yeast (instant),1 tablespoon,9.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",14.786765,0.608652
322,Yogurt,1 cup,227.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236,0.959473
323,Yuletide Cheer Fruit Blend,1 cup,130.0,"IngredientAmount(quantity=Fraction(1, 1), quan...",236.588236,0.549478


The columns are converted to lowercase, and un-necessary columns are dropped, to compress the data and make it easier to work with.

In [19]:
data_df = data_df.rename(columns = {"Ingredient": "ingredient", "Grams": "grams"})
data_df = data_df.drop(["Volume", "parsed amounts"], axis = 1)

In [20]:
data_df

,ingredient,grams,volume (cm^3),density (g/cm^3)
0,'00' Pizza Flour,116.0,236.588236,0.490303
1,Agave syrup,84.0,59.147059,1.420189
2,All-Purpose Baking Mix,120.0,236.588236,0.507210
3,All-Purpose Flour,120.0,236.588236,0.507210
4,Almond butter,68.0,59.147059,1.149677
...,...,...,...,...
320,Yeast (instant),7.0,11.090074,0.631195
321,Yeast (instant),9.0,14.786765,0.608652
322,Yogurt,227.0,236.588236,0.959473
323,Yuletide Cheer Fruit Blend,130.0,236.588236,0.549478


In [21]:
data_df.to_csv("ingredient_densities.csv", index = False)